# Week 3 — Imbalanced Classification & LightGBM Modeling
## Member 3 — Context & Integration

## 1. Environment Setup & Imports
## 2. Load Fused Dataset
## 3. Feature Matrix Extraction
## 4. NaN Verification
## 5. Class Distribution

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder

print("All imports successful")

## 2. Load Fused Dataset

In [ ]:
df = pd.read_csv('../data/ai4i2020.csv')

np.random.seed(42)
df['ambient_temp_C'] = np.random.normal(loc=28, scale=5, size=len(df))
df['factory_load_pct'] = np.random.uniform(50, 100, size=len(df))
df['humidity_pct'] = np.random.normal(loc=60, scale=10, size=len(df))

le = LabelEncoder()
df['Type_enc'] = le.fit_transform(df['Type'])

print("Fused dataset loaded!")
print("Shape:", df.shape)

In [ ]:
ext_features = [
    'Air temperature [K]', 'Process temperature [K]',
    'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]',
    'Type_enc', 'ambient_temp_C', 'factory_load_pct', 'humidity_pct'
]

X = df[ext_features]
y = df['Machine failure']

print("Feature Matrix X shape:", X.shape)
print("Target y shape:", y.shape)

In [ ]:
print("NaN check:")
print(X.isnull().sum().sum())

if X.isnull().sum().sum() == 0:
    print("✅ No NaN values — dataset is clean!")

In [ ]:
print("Class Distribution:")
print(y.value_counts())
print(f"\nNo Failure: {(y==0).sum()} — {(y==0).mean()*100:.2f}%")
print(f"Failure: {(y==1).sum()} — {(y==1).mean()*100:.2f}%")
print(f"Imbalance Ratio: {(y==0).sum()/(y==1).sum():.1f}:1")
print("\n✅ Dataset ready for SMOTE + LightGBM modeling!")

## Commentary — Feature Matrix & Class Distribution

- Total features: 9 (5 internal sensors + Type_enc + 3 external context)
- Total samples: 10,000
- No NaN values — dataset clean and ready
- Imbalance Ratio: 28.5:1 — SMOTE will be applied inside training folds only

## 6. SMOTE Validation — No Data Leakage Check

In [ ]:
from sklearn.model_selection import StratifiedKFold
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from lightgbm import LGBMClassifier

# Setup 5-fold stratified cross validation
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Test SMOTE on first fold only to validate
for fold, (train_idx, test_idx) in enumerate(skf.split(X, y)):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
    
    print(f"=== Fold {fold+1} ===")
    print(f"Training set size: {len(X_train)}")
    print(f"Test set size: {len(X_test)}")
    print(f"\nClass counts BEFORE SMOTE (training only):")
    print(y_train.value_counts())
    
    # Apply SMOTE only on training data
    smote = SMOTE(random_state=42)
    X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)
    
    print(f"\nClass counts AFTER SMOTE (training only):")
    import pandas as pd
    print(pd.Series(y_train_resampled).value_counts())
    
    print(f"\nTest set class counts (UNCHANGED — no leakage):")
    print(y_test.value_counts())
    
    # Assert SMOTE only applied to training data
    assert len(X_test) == len(y_test), "Test set size mismatch!"
    assert y_test.value_counts()[1] < y_test.value_counts()[0], "Test set still imbalanced — no leakage confirmed!"
    
    print(f"\n✅ Fold {fold+1}: SMOTE applied correctly — NO DATA LEAKAGE!")
    break  # Only test first fold for validation

## Commentary — SMOTE Validation

SMOTE validation confirms:

- **SMOTE is applied ONLY on training data** inside each fold — never on test data
- **Before SMOTE:** Training set is imbalanced (28.5:1 ratio)
- **After SMOTE:** Training set is balanced (1:1 ratio) — equal failure and no-failure samples
- **Test set remains unchanged** — still imbalanced, reflecting real-world distribution
- **No data leakage confirmed** — future test samples never influence the training process

This is critical for getting honest model evaluation metrics. A common mistake is applying SMOTE before splitting — this would let synthetic samples "leak" information about the test set into training, inflating performance metrics artificially.

## 7. LightGBM Training & SHAP TreeExplainer

In [ ]:
from sklearn.model_selection import train_test_split
from lightgbm import LGBMClassifier
import shap

# Split into 80% train, 20% test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set: {X_train.shape}")
print(f"Test set: {X_test.shape}")
print(f"\nTraining class distribution:")
print(y_train.value_counts())

In [ ]:
# Fix feature names — remove special characters for LightGBM
import re

def clean_feature_names(df):
    df.columns = [re.sub(r'[^A-Za-z0-9_]+', '_', col) for col in df.columns]
    return df

X_train_resampled_df = pd.DataFrame(X_train_resampled, columns=ext_features)
X_test_df = pd.DataFrame(X_test, columns=ext_features)

X_train_resampled_df = clean_feature_names(X_train_resampled_df)
X_test_df = clean_feature_names(X_test_df)

print("Cleaned feature names:")
print(list(X_train_resampled_df.columns))

In [ ]:
# Apply SMOTE on training set
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

print(f"After SMOTE — Training set shape: {X_train_resampled.shape}")
print(f"Class distribution after SMOTE:")
print(pd.Series(y_train_resampled).value_counts())

# Train LightGBM model
# Train LightGBM model with cleaned feature names
lgbm = LGBMClassifier(random_state=42, n_jobs=-1, verbose=-1)
lgbm.fit(X_train_resampled_df, y_train_resampled)
print("\n✅ LightGBM model trained successfully!")

In [ ]:
# Compute SHAP TreeExplainer
explainer = shap.TreeExplainer(lgbm)
shap_values = explainer.shap_values(X_test_df)

if isinstance(shap_values, list):
    shap_vals = shap_values[1]
else:
    shap_vals = shap_values

mean_shap = np.abs(shap_vals).mean(axis=0)
shap_importance = pd.Series(mean_shap, index=X_train_resampled_df.columns).sort_values(ascending=False)

print("Top 5 features by mean absolute SHAP value:")
print(shap_importance.head(5))

## 8. Tuning scale_pos_weight for Imbalance Handling

In [ ]:
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import f1_score, make_scorer
from lightgbm import LGBMClassifier
import re

# Clean feature names
X_clean = X.copy()
X_clean.columns = [
    re.sub(r'_+', '_',
           re.sub(r'[^A-Za-z0-9_]', '_', str(col)))
    for col in X.columns
]

# Define scorer
macro_f1_scorer = make_scorer(f1_score, average='macro')
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Test different scale_pos_weight values
scale_pos_weights = [None, 10, 20, 50]
results = {}

for spw in scale_pos_weights:

    if spw is None:
        lgbm_test = LGBMClassifier(
            random_state=42,
            n_jobs=-1,
            verbose=-1
        )
        label = "Default (no weight)"
    else:
        lgbm_test = LGBMClassifier(
            random_state=42,
            n_jobs=-1,
            verbose=-1,
            scale_pos_weight=spw
        )
        label = f"scale_pos_weight={spw}"

    scores = cross_val_score(
        lgbm_test,
        X_clean,
        y,
        cv=skf,
        scoring=macro_f1_scorer
    )

    results[label] = scores.mean()

    print(
        f"{label}: Macro F1 = {scores.mean():.4f} "
        f"(+/- {scores.std():.4f})"
    )

print("\nBest configuration:")
best = max(results, key=results.get)
print(f"{best}: {results[best]:.4f}")

## Commentary — scale_pos_weight Tuning

`scale_pos_weight` tells LightGBM to penalize misclassifying the minority class (failures) more heavily.

**Results Analysis:**
- Higher `scale_pos_weight` increases recall for failure class but may reduce precision
- The optimal value balances between catching real failures and avoiding false alarms
- Macro F1 score is used as the primary metric since it equally weights both classes

**Conclusion:**
The best `scale_pos_weight` configuration will be used in the final model pipeline in Day 4.